# LDA 설정 민감도 분석 재집계

- 질문: 토픽 수·불용어·난수 시드를 바꾸어도 원고의 한국어 및 중국어 비교가 유지되는가?
- 입력: 고정 말뭉치, 기존 LDA 배정, 12회 학습의 문서 배정 및 상위 단어.
- 범위: 저장된 모델 출력에서 다시 계산한다. 이 노트북은 모델을 재학습하지 않는다. 재학습은 `python analysis/scripts/20_lda_sensitivity.py`로 수행한다.
- 대응: 본문 4.1과 4.2. 기존 토픽 번호와 새 모델의 토픽 번호는 직접 대응하지 않는다.

## 환경과 입력

- 계산: 입력 해시·문서 키·집합 크기를 확인한다.
- 입력: 말뭉치와 설정별 문서 배정.
- 주의: 2026년은 1~9월이다. 언어 집합의 분모를 유지한다.

In [ ]:
from pathlib import Path
import importlib.util, json, sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'verification').exists())
spec=importlib.util.spec_from_file_location('sensitivity', ROOT/'analysis/scripts/20_lda_sensitivity.py')
m=importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
paths, corpus, old, topics, old_reference=m.inputs()
preserved=json.loads(m.OUT.read_text())
records=json.loads(m.RUNS.read_text())
assert len(records)==12
assert all(r['ids']==corpus.id.tolist() for r in records)
assert len(corpus)==20107
print(sys.version.split()[0], {l:int(corpus[f'lang_{l}'].sum()) for l in ['ko','zh','ja']})

## 저장된 출력에서 재집계

- 계산: 상위 단어와 문서집합의 Jaccard 유사도, 집합별 편수, 연도별 비중을 다시 계산한다.
- 입력: 12개 설정의 문서 배정과 상위 단어.
- 주의: 공통 분석 함수를 호출하는 실행 재현이며 독립 재검산과 구별한다. 보존한 JSON은 덮어쓰지 않는다.

In [ ]:
recomputed=m.summarize(records)
assert recomputed == preserved
baseline=next(r for r in recomputed['runs'] if r['name']=='current_k25_s42')
assert baseline['baseline_reproduction']=={'assignment_mismatches':0,'top_word_mismatches':0}
print('12개 설정 재집계 및 보존 정본 대조 통과')

## 독립 집계 확인

- 계산: 선택된 대응 토픽의 편수와 연도 비중을 pandas 교차표로 별도 계산한다.
- 입력: 설정별 문서 배정과 원고 집합 라벨.
- 주의: 토픽 대응 알고리즘의 독립 재구현은 포함하지 않는다. 편수·분모·연도 비중을 별도 계산하여 확인한다.

In [ ]:
checks=0
for record, result in zip(records,recomputed['runs']):
    assert record['name']==result['name']
    frame=corpus[['id','year','lang_ko','lang_zh','lang_ja']].copy()
    frame['topic']=record['labels']
    counts=frame.groupby('topic').size()
    years=pd.crosstab(frame.topic,frame.year)
    for target in result['targets'].values():
        for match in target['matches'].values():
            topic=match['topic']
            assert int(counts[topic])==match['overall_n']; checks+=1
            for lang in ['ko','zh','ja']:
                count=int(frame.loc[frame[f'lang_{lang}']].topic.eq(topic).sum())
                assert count==match['language_n'][lang]; checks+=1
            for year in m.YEARS:
                count=int(years.loc[topic,year]); denom=int(frame.year.eq(year).sum())
                assert count==match['year_n'][str(year)]
                assert abs(100*count/denom-match['year_share_pct'][str(year)])<1e-12; checks+=2
print('독립 집계 확인:',checks,'개 항목')

## 대응 토픽 비교

- 계산: 원고의 세 개 토픽과 단어가 가장 많이 겹치는 토픽의 비교표를 표시한다.
- 입력: 재집계 결과.
- 주의: Jaccard 유사도가 낮거나 문서 대응과 다른 경우 토픽 동일성이 확보된 것으로 보지 않는다. 0편의 유지 여부는 대응의 질과 함께 읽는다.

In [ ]:
rows=[]
for r in recomputed['runs']:
    for name,t in r['targets'].items():
        v=t['matches']['top_words']
        rows.append({'설정':r['name'],'기준 토픽':name,'대응 단어':'·'.join(v['top_words'][:2]),
                     '단어 Jaccard':round(v['word_jaccard'],3),'문서 Jaccard':round(v['document_jaccard'],3),
                     '두 대응 일치':t['matching_methods_agree'],'한국어 편수':v['language_n']['ko'],
                     '중국어 편수':v['language_n']['zh'],'한국어 비율비':round(v['ko_share_ratio_to_overall'],2),
                     '2026-2023 비중 차이':round(v['change_2026_minus_2023_pp'],2)})
pd.DataFrame(rows)

## 해석 범위

- 토픽 수가 줄어들면 기존 토픽들이 합쳐질 수 있다. 단어가 가장 많이 겹치는 토픽을 같은 연구 주제의 확정된 경계로 취급하지 않는다.
- 비교 결과는 시험한 12개 설정에 한정된다. 한국어 25편의 표집 규모와 최초 수집 범위는 이 분석으로 바뀌지 않는다.
- 중국어 retrieval·graph 13편의 내용 검토는 별도 `notes/LDA_RETRIEVAL_ABSTRACT_REVIEW_20261015.md`에 있다. AI 보조 검토이며 사람 판정을 대체하지 않는다.